# vcloset · CatVTON Inference Server (Colab Free T4)

이 노트북은 Google Colab 무료 T4 GPU에서 CatVTON 추론 서버를 띄우고, ngrok 으로 외부에 노출시킵니다.
맥북 로컬에서 돌리는 vcloset Next.js 앱이 이 ngrok URL 을 통해 합성을 호출합니다.

## 사전 준비
1. **Runtime → Change runtime type → T4 GPU** 로 설정
2. **ngrok 무료 가입** → https://dashboard.ngrok.com/get-started/your-authtoken 에서 authtoken 복사
3. 아래 셀의 `NGROK_AUTHTOKEN` 에 붙여넣기

## 실행 순서
맨 위부터 순서대로 한 셀씩 실행 (Shift+Enter). 마지막 셀에서 https 주소가 출력되면 그걸 맥북 `.env` 의 `INFERENCE_URL` 에 넣으세요.

In [ ]:
# 1) GPU 확인
!nvidia-smi

In [ ]:
# 2) 시스템 패키지 + 파이썬 의존성 설치 (~3-5분)
!pip install -q torch==2.4.0 torchvision diffusers==0.31.0 transformers==4.45.2 accelerate \
    huggingface_hub==0.25.2 fastapi uvicorn pyngrok nest-asyncio pillow opencv-python-headless \
    einops omegaconf cloudpickle scipy fvcore iopath pycocotools yacs timm scikit-image av==12.3.0

In [ ]:
# 3) CatVTON 저장소 클론
import os, sys
if not os.path.isdir('CatVTON'):
    !git clone https://github.com/Zheng-Chong/CatVTON.git
sys.path.insert(0, '/content/CatVTON')
!ls CatVTON | head

In [ ]:
# 4) 사전학습 weight 다운로드 (HuggingFace, 약 7GB, ~5-8분)
# .pkl/.pth + DensePose/SCHP 하위 디렉토리까지 포함해야 AutoMasker 가 동작합니다.
from huggingface_hub import snapshot_download
CKPT = snapshot_download(
    repo_id='zhengchong/CatVTON',
    local_dir='/content/ckpt',
    allow_patterns=['*.bin','*.json','*.txt','*.safetensors','*.yaml','*.pkl','*.pth','DensePose/*','SCHP/*'],
)
print('done →', CKPT)
!ls /content/ckpt/DensePose /content/ckpt/SCHP 2>/dev/null | head

In [ ]:
# 5) 모델 로드 — CatVTON 파이프라인 초기화 (FP16, T4 호환)
import torch
from diffusers.image_processor import VaeImageProcessor
from model.pipeline import CatVTONPipeline  # CatVTON 저장소 내부 모듈
from model.cloth_masker import AutoMasker

DEVICE = 'cuda'
DTYPE = torch.float16
BASE_SD = 'booksforcharlie/stable-diffusion-inpainting'  # CatVTON 권장 base

pipeline = CatVTONPipeline(
    base_ckpt=BASE_SD,
    attn_ckpt='/content/ckpt',
    attn_ckpt_version='mix',
    weight_dtype=DTYPE,
    use_tf32=True,
    device=DEVICE,
)
masker = AutoMasker(densepose_ckpt='/content/ckpt/DensePose', schp_ckpt='/content/ckpt/SCHP', device=DEVICE)
print('CatVTON loaded')

In [ ]:
# 6) FastAPI 서버 정의
import base64, io, traceback
from PIL import Image
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title='vcloset CatVTON server')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

class TryOnReq(BaseModel):
    person_b64: str
    garment_b64: str
    cloth_type: str = 'upper'  # upper | lower | overall
    num_steps: int = 30
    guidance: float = 2.5

def b64_to_pil(b64: str) -> Image.Image:
    if ',' in b64:
        b64 = b64.split(',', 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64))).convert('RGB')

def pil_to_b64(img: Image.Image) -> str:
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode('ascii')

@app.get('/health')
def health():
    return {'ok': True, 'device': DEVICE}

@app.post('/try-on')
def try_on(req: TryOnReq):
    try:
        person = b64_to_pil(req.person_b64).resize((768, 1024))
        garment = b64_to_pil(req.garment_b64).resize((768, 1024))
        mask = masker(person, req.cloth_type)['mask']
        result = pipeline(
            image=person,
            condition_image=garment,
            mask=mask,
            num_inference_steps=req.num_steps,
            guidance_scale=req.guidance,
        )[0]
        return {'result_b64': pil_to_b64(result)}
    except Exception as e:
        traceback.print_exc()
        return {'error': str(e)}

print('FastAPI app ready')

In [ ]:
# 7) ngrok 터널 + uvicorn 실행 (스레드 — Colab event loop 충돌 회피)
# ← 출력되는 https URL 을 맥북 .env 의 INFERENCE_URL 에 입력
import threading, uvicorn
from pyngrok import ngrok, conf

NGROK_AUTHTOKEN = ''  # ← 여기에 ngrok dashboard 의 authtoken 붙여넣기
assert NGROK_AUTHTOKEN, 'ngrok authtoken 을 먼저 입력하세요'
conf.get_default().auth_token = NGROK_AUTHTOKEN

# 재실행 시 기존 터널 정리
try:
    for t in ngrok.get_tunnels(): ngrok.disconnect(t.public_url)
except Exception: pass

tunnel = ngrok.connect(8000, 'http')
print('\n========================================')
print('   INFERENCE_URL =', tunnel.public_url)
print('========================================\n')

def _serve():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')

threading.Thread(target=_serve, daemon=True).start()
print('uvicorn started in background. 셀이 곧 끝나도 서버는 계속 돌아갑니다.')